# Download Backup

In [9]:
#!/usr/bin/env python3
"""
Downloads BoloChinese's submission data one request at a time, bypassing the
bulk zip-backup endpoint entirely. Uses the same admin API the dashboard
uses - just one small request per file instead of one big zip, so a slow
connection or a large dataset shows real, granular progress instead of a
long silent wait.

Requires: pip install requests (optional: python-dotenv, to auto-load .env)

Set BASE_URL / ADMIN_EMAIL / ADMIN_PASSWORD as env vars, or drop a .env
file next to this script (see .env.example), then just run:
    python3 backup.py

Writes, under BACKUP_DIR:
    submissions.csv                          one row per submission
    <project>/<username>/<dialogueId>.wav    one file per recording

Safe to re-run: existing audio files are skipped, not re-downloaded.
"""
import csv
import os
import re
import sys
import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

BASE_URL = os.environ.get("BASE_URL", "")  # no trailing slash; e.g.
                                            # "https://your-app.example.com/api"
ADMIN_EMAIL = os.environ.get("ADMIN_EMAIL", "")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD", "")

BACKUP_DIR = "bolochinese_backup"
PAGE_SIZE = 100  # max the API allows
REQUEST_TIMEOUT = 60

CSV_COLUMNS = [
    "project", "username", "email", "dialogueId", "taskId", "status",
    "chineseTranscript", "pinyin", "correctedChineseTranscript", "correctedPinyin",
    "editCharCount", "pinyinVerified", "isCorrected", "discarded", "discardedAt",
    "audioDurationSeconds", "audioFileSizeBytes", "timeSpentMs",
    "createdAt", "updatedAt", "audioFile",
]


def sanitize(name):
    name = re.sub(r'[\\/:*?"<>|\x00-\x1f]+', "_", str(name or "").strip())
    return name or "unnamed"


def login(session):
    r = session.post(
        f"{BASE_URL}/auth/login",
        json={"email": ADMIN_EMAIL, "password": ADMIN_PASSWORD},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    session.headers["Authorization"] = f"Bearer {r.json()['data']['token']}"


def get_projects(session):
    r = session.get(f"{BASE_URL}/admin/projects", timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()["data"]


def get_submissions_page(session, project_id, page):
    r = session.get(
        f"{BASE_URL}/admin/projects/{project_id}/submissions",
        params={"page": page, "limit": PAGE_SIZE},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    return r.json()["data"]


def download_audio(session, submission_id, dest_path):
    r = session.get(
        f"{BASE_URL}/admin/submissions/{submission_id}/audio",
        stream=True,
        timeout=REQUEST_TIMEOUT,
    )
    if r.status_code == 404:
        return False
    r.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
    return True


def main():
    if not BASE_URL or not ADMIN_EMAIL or not ADMIN_PASSWORD:
        sys.exit(
            "Set BASE_URL / ADMIN_EMAIL / ADMIN_PASSWORD - as env vars, or in a "
            ".env file next to this script (see .env.example)."
        )

    os.makedirs(BACKUP_DIR, exist_ok=True)
    session = requests.Session()
    login(session)

    projects = get_projects(session)
    print(f"Found {len(projects)} project(s).")

    rows = []
    audio_jobs = []  # (submission_id, dest_path)

    for project in projects:
        page = 1
        while True:
            data = get_submissions_page(session, project["_id"], page)
            items = data["items"]
            for item in items:
                task = item.get("taskId") or {}
                user = item.get("userId") or {}
                audio = item.get("audio") or {}
                discarded = item.get("discarded") or {}
                dialogue_id = task.get("dialogueId") or item["_id"]
                username = sanitize(user.get("username") or user.get("email") or "unknown")

                audio_rel_path = ""
                if audio.get("url"):
                    audio_rel_path = os.path.join(
                        sanitize(project["name"]), username, f"{sanitize(dialogue_id)}.wav"
                    )
                    audio_jobs.append((item["_id"], os.path.join(BACKUP_DIR, audio_rel_path)))

                rows.append({
                    "project": project["name"],
                    "username": user.get("username") or "",
                    "email": user.get("email") or "",
                    "dialogueId": dialogue_id,
                    "taskId": task.get("taskId") or "",
                    "status": item.get("status") or "",
                    "chineseTranscript": task.get("chineseTranscript") or "",
                    "pinyin": task.get("pinyin") or "",
                    "correctedChineseTranscript": item.get("correctedChineseTranscript") or "",
                    "correctedPinyin": item.get("correctedPinyin") or "",
                    "editCharCount": item.get("editCharCount") or 0,
                    "pinyinVerified": item.get("pinyinVerified"),
                    "isCorrected": item.get("isCorrected"),
                    "discarded": discarded.get("flagged", False),
                    "discardedAt": discarded.get("discardedAt") or "",
                    "audioDurationSeconds": audio.get("durationSeconds") or "",
                    "audioFileSizeBytes": audio.get("fileSizeBytes") or "",
                    "timeSpentMs": item.get("timeSpentMs") or 0,
                    "createdAt": item.get("createdAt") or "",
                    "updatedAt": item.get("updatedAt") or "",
                    "audioFile": audio_rel_path,
                })

            total_pages = data["pagination"]["totalPages"]
            print(f"  {project['name']}: page {page}/{total_pages} ({len(items)} rows)")
            if page >= total_pages:
                break
            page += 1

    csv_path = os.path.join(BACKUP_DIR, "submissions.csv")
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nWrote {len(rows)} row(s) to {csv_path}")

    total = len(audio_jobs)
    print(f"\nDownloading {total} audio file(s), one at a time...")
    ok, skipped, failed = 0, 0, []
    for i, (submission_id, dest_path) in enumerate(audio_jobs, start=1):
        pct = round(i / total * 100) if total else 100
        label = f"  [{i}/{total}] ({pct}%) {os.path.basename(dest_path)} ... "
        if os.path.exists(dest_path):
            print(label + "skip (already downloaded)")
            skipped += 1
            continue
        print(label, end="", flush=True)
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        try:
            if download_audio(session, submission_id, dest_path):
                print("OK")
                ok += 1
            else:
                print("no audio")
        except Exception as e:
            print(f"FAILED: {e}")
            failed.append((submission_id, dest_path, str(e)))

    print(f"\nDone. {ok} downloaded, {skipped} already had a copy, {len(failed)} failed (of {total}).")
    if failed:
        print("Failed:")
        for submission_id, dest_path, err in failed:
            print(f"  {submission_id} -> {dest_path}: {err}")


if __name__ == "__main__":
    main()

Found 9 project(s).
  vikashsaini: page 1/1 (0 rows)
  ankurmaurya: page 1/5 (100 rows)
  ankurmaurya: page 2/5 (100 rows)
  ankurmaurya: page 3/5 (100 rows)
  ankurmaurya: page 4/5 (100 rows)
  ankurmaurya: page 5/5 (17 rows)
  rishikeshmeena: page 1/2 (100 rows)
  rishikeshmeena: page 2/2 (4 rows)
  juhikumari: page 1/1 (27 rows)
  vedantmaurya: page 1/1 (20 rows)
  somyadutta: page 1/1 (85 rows)
  surajpegu: page 1/1 (40 rows)
  gudiyasingh: page 1/1 (43 rows)
  utakarshshukla: page 1/1 (16 rows)

Wrote 752 row(s) to bolochinese_backup/submissions.csv


Done. 0 downloaded, 0 already had a copy, 0 failed (of 0).


# Verify

In [10]:
#!/usr/bin/env python3
"""
Verifies a local backup (from download_backup.py) still matches what the
server/Cloudinary currently has - and tells you clearly *why* when it
doesn't, rather than just pass/fail.

For every row in submissions.csv with an audioFile:
  1. Confirms the local .wav exists and its size matches what was recorded
     at download time (catches a truncated/corrupted download).
  2. Re-fetches that submission live from the server right now and compares:
       - Cloudinary no longer has the audio (already cleaned up) -> expected,
         reported as OK. Your local file is now the only copy that exists.
       - Cloudinary still has it, but the size differs from your local copy
         -> the submission changed (re-recorded) since you backed it up.
       - not found at all -> the project/task was deleted since your backup.

Doesn't touch Cloudinary directly - it verifies against the app's own API,
which already reflects Cloudinary's real state, so no Cloudinary credentials
are needed here.

Requires: pip install requests (optional: python-dotenv, to auto-load .env)

Set BASE_URL / ADMIN_EMAIL / ADMIN_PASSWORD as env vars, or drop a .env
file next to this script (see .env.example). Edit BACKUP_DIR below if
you changed it from the default, then run:
    python3 verify.py
"""

# Cloudinary re-packages the WAV container on upload, so the size it actually
# stores/serves can be a handful of bytes off from the "audioFileSizeBytes"
# recorded in the database at upload time (which was the size of the raw
# buffer *before* Cloudinary touched it) - not corruption. A real truncated
# or corrupted download will be off by far more than this. Only flag a
# difference bigger than the tolerance.
SIZE_TOLERANCE_BYTES = 256


def login(session):
    r = session.post(
        f"{BASE_URL}/auth/login",
        json={"email": ADMIN_EMAIL, "password": ADMIN_PASSWORD},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    session.headers["Authorization"] = f"Bearer {r.json()['data']['token']}"


def get_projects(session):
    r = session.get(f"{BASE_URL}/admin/projects", timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()["data"]


def get_all_submissions(session, project_id):
    """Every submission for a project, fetched fresh right now - includes
    ones whose audio has already been purged (they just show audio: null)."""
    items = []
    page = 1
    while True:
        r = session.get(
            f"{BASE_URL}/admin/projects/{project_id}/submissions",
            params={"page": page, "limit": PAGE_SIZE},
            timeout=REQUEST_TIMEOUT,
        )
        r.raise_for_status()
        data = r.json()["data"]
        items.extend(data["items"])
        if page >= data["pagination"]["totalPages"]:
            break
        page += 1
    return items


def main():
    if not BASE_URL or not ADMIN_EMAIL or not ADMIN_PASSWORD:
        sys.exit(
            "Set BASE_URL / ADMIN_EMAIL / ADMIN_PASSWORD - as env vars, or in a "
            ".env file next to this script (see .env.example)."
        )

    csv_path = os.path.join(BACKUP_DIR, "submissions.csv")
    if not os.path.exists(csv_path):
        sys.exit(f"Can't find {csv_path} - run download_backup.py first.")

    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))

    audio_rows = [r for r in rows if r.get("audioFile")]
    print(f"Checking {len(audio_rows)} audio file(s) from {csv_path}...\n")

    session = requests.Session()
    login(session)

    projects = get_projects(session)
    print(f"Fetching current server state for {len(projects)} project(s)...")

    # (project name, username, dialogueId) -> live submission item. Uses the
    # same "username or empty string" rule download_backup.py used for the
    # CSV column (not the fuller "or email or unknown" one it used for
    # filenames), so the keys line up with what's actually in the CSV.
    live_by_key = {}
    for project in projects:
        for item in get_all_submissions(session, project["_id"]):
            task = item.get("taskId") or {}
            user = item.get("userId") or {}
            dialogue_id = task.get("dialogueId") or item["_id"]
            username = user.get("username") or ""
            live_by_key[(project["name"], username, dialogue_id)] = item

    print("Done fetching. Comparing...\n")

    ok, purged, stale, missing_local, size_mismatch, not_found = [], [], [], [], [], []

    for row in audio_rows:
        label = f"{row['project']}/{row['username']}/{row['dialogueId']}"
        local_path = os.path.join(BACKUP_DIR, row["audioFile"])

        if not os.path.exists(local_path):
            missing_local.append(label)
            print(f"  ✗ MISSING LOCALLY: {label}")
            continue

        local_size = os.path.getsize(local_path)
        recorded_size = int(row.get("audioFileSizeBytes") or 0)
        if recorded_size and abs(local_size - recorded_size) > SIZE_TOLERANCE_BYTES:
            size_mismatch.append(label)
            print(f"  ✗ CORRUPTED DOWNLOAD: {label} (disk has {local_size} bytes, CSV recorded {recorded_size})")
            continue

        key = (row["project"], row["username"], row["dialogueId"])
        live = live_by_key.get(key)
        if live is None:
            not_found.append(label)
            print(f"  ✗ NOT FOUND ON SERVER: {label} (project/task deleted since backup?)")
            continue

        live_audio = live.get("audio") or {}
        if not live_audio.get("url"):
            purged.append(label)
            print(f"  ✓ already cleaned up on Cloudinary: {label} (your local copy is now the only one)")
            continue

        live_size = live_audio.get("fileSizeBytes") or 0
        if live_size and abs(live_size - local_size) > SIZE_TOLERANCE_BYTES:
            stale.append(label)
            print(f"  ⚠ CHANGED SINCE BACKUP: {label} (Cloudinary now has {live_size} bytes, your copy has {local_size})")
            continue

        ok.append(label)
        print(f"  ✓ matches: {label}")

    print(
        f"\nDone. {len(ok)} match, {len(purged)} already cleaned up (local copy is the only one), "
        f"{len(stale)} changed since backup, {len(missing_local)} missing locally, "
        f"{len(size_mismatch)} corrupted download, {len(not_found)} not found on server "
        f"(of {len(audio_rows)} total)."
    )

    problems = missing_local + size_mismatch + not_found + stale
    if problems:
        print(f"\n{len(problems)} item(s) need attention - see the ✗/⚠ lines above.")
        sys.exit(1)
    print("\nEverything checks out.")


if __name__ == "__main__":
    main()

Checking 0 audio file(s) from bolochinese_backup/submissions.csv...

Fetching current server state for 9 project(s)...
Done fetching. Comparing...


Done. 0 match, 0 already cleaned up (local copy is the only one), 0 changed since backup, 0 missing locally, 0 corrupted download, 0 not found on server (of 0 total).

Everything checks out.
